In [1]:
# import torch
# import time
# from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
# from peft import PeftModel

# # ============================================================
# # PATHS
# # ============================================================
# base_model_dir = "SLM_Models/SmolLM2-360M-Instruct"
# adapter_dir = "final_adapter_smollm5"

# # ============================================================
# # STEP 1: Load tokenizer
# # ============================================================
# print("⏳ Loading tokenizer...")
# tokenizer = AutoTokenizer.from_pretrained(adapter_dir, local_files_only=True)
# tokenizer.pad_token = tokenizer.eos_token
# print("✅ Tokenizer loaded!")

# # ============================================================
# # STEP 2: Load base model in 4-bit
# # ============================================================
# print("⏳ Loading base model (4-bit)...")
# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_quant_type="nf4",
#     bnb_4bit_compute_dtype=torch.float16,
#     bnb_4bit_use_double_quant=True
# )

# base_model = AutoModelForCausalLM.from_pretrained(
#     base_model_dir,
#     quantization_config=bnb_config,
#     device_map="auto",
#     local_files_only=True
# )
# print(f"✅ Base model loaded! GPU: {torch.cuda.memory_allocated() / (1024**2):.0f} MB")

# # ============================================================
# # STEP 3: Load and merge LoRA adapters
# # ============================================================
# print("⏳ Loading LoRA adapters...")
# model = PeftModel.from_pretrained(base_model, adapter_dir)
# print(f"✅ LoRA adapters merged! GPU: {torch.cuda.memory_allocated() / (1024**2):.0f} MB")

# print("\n" + "=" * 60)
# print("   ✅ FINE-TUNED AutoHeal MODEL READY FOR INFERENCE")
# print("=" * 60)

In [2]:
# import torch
# import time
# from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
# from peft import PeftModel

# # ============================================================
# # PATHS
# # ============================================================
# base_model_dir = "/home/dell/SLM_Engineering/SLM_Models/SmolLM2-360M-Instruct"
# adapter_dir = "/home/dell/SLM_Engineering/V1_Final_INT8_2000_adapter_smollm"

# # ============================================================
# # STEP 1: Load tokenizer
# # ============================================================
# print("⏳ Loading tokenizer...")
# tokenizer = AutoTokenizer.from_pretrained(adapter_dir, local_files_only=True)
# tokenizer.pad_token = tokenizer.eos_token
# print("✅ Tokenizer loaded!")

# # ============================================================
# # STEP 2: Load base model in 8-bit
# # ============================================================
# print("⏳ Loading base model (8-bit)...")
# bnb_config = BitsAndBytesConfig(
#     load_in_8bit=True,
#     bnb_8bit_compute_dtype=torch.float16
# )

# base_model = AutoModelForCausalLM.from_pretrained(
#     base_model_dir,
#     quantization_config=bnb_config,
#     device_map="auto",
#     local_files_only=True
# )
# print(f"✅ Base model loaded! GPU: {torch.cuda.memory_allocated() / (1024**2):.0f} MB")

# # ============================================================
# # STEP 3: Load and merge LoRA adapters
# # ============================================================
# print("⏳ Loading LoRA adapters...")
# model = PeftModel.from_pretrained(base_model, adapter_dir)
# print(f"✅ LoRA adapters merged! GPU: {torch.cuda.memory_allocated() / (1024**2):.0f} MB")

# print("\n" + "=" * 60)
# print("   ✅ FINE-TUNED AutoHeal MODEL READY FOR INFERENCE")
# print("=" * 60)

In [3]:
# Updated INT8 LoRA testing code


import torch
import time
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

# ============================================================
# PATHS
# ============================================================
base_model_dir = "/home/dell/SLM_Engineering/SLM_Models/SmolLM2-360M-Instruct"
adapter_dir = "/home/dell/SLM_Engineering/V1_Final_INT8_2000_adapter_smollm"

# ============================================================
# STEP 1: Load tokenizer
# ============================================================
print("⏳ Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    adapter_dir,
    local_files_only=True
)

tokenizer.pad_token = tokenizer.eos_token

print("✅ Tokenizer loaded!")

# ============================================================
# STEP 2: Load base model in 8-bit
# ============================================================
print("⏳ Loading base model (INT8)...")

bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_threshold=6.0,
    llm_int8_enable_fp32_cpu_offload=False
)

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_dir,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    local_files_only=True
)

print(
    f"✅ INT8 base model loaded! "
    f"GPU: {torch.cuda.memory_allocated() / (1024**2):.0f} MB"
)

# ============================================================
# STEP 3: Load LoRA adapters (without merging)
# ============================================================
print("⏳ Loading LoRA adapters...")

model = PeftModel.from_pretrained(
    base_model,
    adapter_dir,
    is_trainable=False
)

model.eval()

print(
    f"✅ LoRA adapters loaded! "
    f"GPU: {torch.cuda.memory_allocated() / (1024**2):.0f} MB"
)

# ============================================================
# STEP 4: Verify INT8 quantization
# ============================================================
import bitsandbytes as bnb

int8_layers = sum(
    isinstance(module, bnb.nn.Linear8bitLt)
    for module in model.modules()
)

print(f"✅ INT8 layers found: {int8_layers}")

assert int8_layers > 0, "INT8 quantization was not applied!"

# ============================================================
# MODEL READY
# ============================================================
print("\n" + "=" * 60)
print("   ✅ FINE-TUNED AutoHeal INT8 MODEL READY FOR INFERENCE")
print("=" * 60)


⏳ Loading tokenizer...
✅ Tokenizer loaded!
⏳ Loading base model (INT8)...
✅ INT8 base model loaded! GPU: 391 MB
⏳ Loading LoRA adapters...
✅ LoRA adapters loaded! GPU: 424 MB
✅ INT8 layers found: 224

   ✅ FINE-TUNED AutoHeal INT8 MODEL READY FOR INFERENCE


In [4]:

# import torch

# print("PyTorch:", torch.__version__)
# print("CUDA:", torch.version.cuda)

# if torch.cuda.is_available():
#     print("GPU:", torch.cuda.get_device_name(0))
# else:
#     print("GPU: CUDA not available")

# print("\nTesting PyTorch 16-bit (FP16) CUDA execution...")

# try:
#     if not torch.cuda.is_available():
#         raise RuntimeError("CUDA is not available")

#     layer = torch.nn.Linear(
#         in_features=16,
#         out_features=16
#     )

#     # Move layer to GPU and convert weights to FP16
#     layer = layer.to(
#         device="cuda",
#         dtype=torch.float16
#     )

#     x = torch.randn(
#         2, 16,
#         device="cuda",
#         dtype=torch.float16
#     )

#     with torch.no_grad():
#         output = layer(x)

#     # Synchronize CUDA to catch asynchronous errors
#     torch.cuda.synchronize()

#     print("Output shape:", output.shape)
#     print("Output dtype:", output.dtype)
#     print("Weight dtype:", layer.weight.dtype)
#     print("✅ PyTorch FP16 CUDA backend works!")

# except Exception as e:
#     print("❌ PyTorch FP16 CUDA test failed")
#     print("Error:", repr(e))

In [5]:
# import os

# os.environ["USE_TF"] = "0"
# os.environ["USE_FLAX"] = "0"

# import torch

# from transformers import (
#     AutoModelForCausalLM,
#     AutoTokenizer
# )

# from peft import PeftModel

# base_model_dir = "/home/dell/SLM_Engineering/SLM_Models/SmolLM2-360M-Instruct"
# adapter_dir = "/home/dell/SLM_Engineering/Final_FP16_3000_final_adapter_smollm"

# print("⏳ Loading tokenizer...")

# tokenizer = AutoTokenizer.from_pretrained(
#     adapter_dir,
#     local_files_only=True
# )

# tokenizer.pad_token = tokenizer.eos_token

# print("✅ Tokenizer loaded!")

# print("⏳ Loading base model in FP16...")

# base_model = AutoModelForCausalLM.from_pretrained(
#     base_model_dir,
#     torch_dtype=torch.float16,
#     device_map="auto",
#     local_files_only=True
# )

# print("✅ Base model loaded!")

# print("⏳ Loading LoRA adapter...")

# model = PeftModel.from_pretrained(
#     base_model,
#     adapter_dir,
#     is_trainable=False
# )

# model.eval()

# print("✅ LoRA adapter loaded!")

# print(
#     f"GPU memory allocated: "
#     f"{torch.cuda.memory_allocated() / (1024**2):.2f} MB"
# )

# print("\n" + "=" * 60)
# print("✅ FINE-TUNED AutoHeal MODEL READY FOR INFERENCE")
# print("=" * 60)

In [6]:
# import json
# import re

# # ============================================================
# # COMPLETE TOOL REGISTRY — All responses from Tool Outputs [2]
# # ============================================================

# # Diagnostic CONFIRMED responses [2]
# DIAG_CONFIRMED = {
#     "GetACAdapterChargeWattageStatus": "Insufficient AC Adapter charge wattage detected. AC Adapter Charge Wattage Notification has to be displayed",
#     "CheckHeavyBatteryUsageByApps": "High battery usage was detected.",
#     "UpdateDrivers": "Driver update has completed.",
#     "CheckHeavyACAdapterUsageSettings": "Device is primarily used on AC adapter. Remediation is required.",
#     "GetBatteryChargeStateChangesCount": "Frequent AC Power Connect/Disconnect.",
#     "CheckLowEnergySettings": "Low energy settings are disabled."
# }

# # Diagnostic NOT CONFIRMED responses [2]
# DIAG_NOT_CONFIRMED = {
#     "GetACAdapterChargeWattageStatus": "Sufficient AC Adapter charge wattage detected.No need to display AC Adapter Charge Wattage Notification.",
#     "CheckHeavyBatteryUsageByApps": "Battery usage is below the threshold.",
#     "UpdateDrivers": "No drivers found for installation after driver scan.",
#     "CheckHeavyACAdapterUsageSettings": "Device is not primarily used on AC adapter. No need to display Battery charge state notification.",
#     "GetBatteryChargeStateChangesCount": "No frequent AC Power Connect/Disconnect detected.",
#     "CheckLowEnergySettings": "Low energy settings are enabled."
# }

# # Diagnostic ERROR responses [2]
# DIAG_ERROR = {
#     "GetACAdapterChargeWattageStatus": "Adapter wattage data not found in DTP. No need to display AC Adapter Charge Wattage Notification.",
#     "CheckHeavyBatteryUsageByApps": "Some error occurred while executing the script.",
#     "UpdateDrivers": "Failed to scan drivers.",
#     "CheckHeavyACAdapterUsageSettings": "Some error occurred while collecting AC adapter usage data.",
#     "GetBatteryChargeStateChangesCount": "Some error occurred while executing GetBatteryChargeStateChangesCount. No need to display Battery charge state notification.",
#     "CheckLowEnergySettings": "Device is not on battery."
# }

# # Remediation SUCCESS responses [2]
# REM_SUCCESS = {
#     "NotifyAdapterChargeWattageIsNotSufficient": "AC Adapter Charge Wattage Notification displayed successfully to user.",
#     "NotifyHeavyBatteryUsageByApps": "User notified of top three apps consuming battery.",
#     "UpdateBIOSBatteryProfile": "BIOS battery profile has been updated to primarily on AC power.",
#     "NotifyBatteryChargeStateChanges": "Battery Charge State Notification displayed successfully to user.",
#     "SetLowEnergySettings": "All settings for low energy are optimized."
# }

# # Active registry — gets modified per test
# TOOL_RESPONSES = {}

# def execute_tool(tool_name):
#     if tool_name in TOOL_RESPONSES:
#         return json.dumps({"result": TOOL_RESPONSES[tool_name]})
#     return json.dumps({"result": f"Unknown tool: {tool_name}"})

# def setup_fallthrough(first_diag, second_diag, remediation, use_error=False):
#     """Configure registry for fallthrough: first NOT confirmed, second CONFIRMED"""
#     global TOOL_RESPONSES
#     TOOL_RESPONSES = {}
    
#     # First diagnostic — NOT CONFIRMED or ERROR
#     if use_error:
#         TOOL_RESPONSES[first_diag] = DIAG_ERROR[first_diag]
#         print(f"  [SET] {first_diag} -> ERROR: {DIAG_ERROR[first_diag][:60]}...")
#     else:
#         TOOL_RESPONSES[first_diag] = DIAG_NOT_CONFIRMED[first_diag]
#         print(f"  [SET] {first_diag} -> NOT CONFIRMED: {DIAG_NOT_CONFIRMED[first_diag][:60]}...")
    
#     # Second diagnostic — CONFIRMED
#     TOOL_RESPONSES[second_diag] = DIAG_CONFIRMED[second_diag]
#     print(f"  [SET] {second_diag} -> CONFIRMED: {DIAG_CONFIRMED[second_diag][:60]}...")
    
#     # Remediation — SUCCESS
#     TOOL_RESPONSES[remediation] = REM_SUCCESS[remediation]
#     print(f"  [SET] {remediation} -> SUCCESS: {REM_SUCCESS[remediation][:60]}...")
    
#     # For UpdateDrivers which is both diagnostic and remediation
#     if second_diag == "UpdateDrivers":
#         TOOL_RESPONSES["UpdateDrivers"] = DIAG_CONFIRMED["UpdateDrivers"]
#         print(f"  [SET] UpdateDrivers serves as both diagnostic+remediation")

# print("Cell 1: Tool Registry loaded")

In [7]:

import json
import re

# ============================================================
# COMPLETE TOOL REGISTRY — Battery Not Charging L2
# Exact responses from Tool Outputs Mapping
# ============================================================

# ============================================================
# Diagnostic CONFIRMED responses
# ============================================================

DIAG_CONFIRMED = {
    "GetACAdapterChargeWattageStatus":
        "Insufficient AC Adapter charge wattage detected. AC Adapter Charge Wattage Notification has to be displayed",

    "CheckBatteryChargeCap":
        "Battery charge cap is enabled."
}

# ============================================================
# Diagnostic NOT CONFIRMED responses
# ============================================================

DIAG_NOT_CONFIRMED = {
    "GetACAdapterChargeWattageStatus":
        "Sufficient AC Adapter charge wattage detected.No need to display AC Adapter Charge Wattage Notification.",

    "CheckBatteryChargeCap":
        "Battery charge cap is disabled."
}

# ============================================================
# Diagnostic ERROR responses
# ============================================================

DIAG_ERROR = {
    "GetACAdapterChargeWattageStatus":
        "Adapter wattage data not found in DTP. No need to display AC Adapter Charge Wattage Notification.",

    "CheckBatteryChargeCap":
        "Error checking battery charge cap status."
}

# # Additional diagnostic error response
# DIAG_ADDITIONAL_ERROR = {
#     "GetACAdapterChargeWattageStatus":
#         "Some error occurred while executing GetACAdapterChargeWattageStatus. No need to display AC Adapter Charge Wattage Notification."
# }

# ============================================================
# Remediation SUCCESS responses
# ============================================================

REM_SUCCESS = {
    "NotifyAdapterChargeWattageIsNotSufficient":
        "AC Adapter Charge Wattage Notification displayed successfully to user.",

    "SetBatteryChargeCap":
        "BIOS battery charge level settings were updated. Remediation was applied."
}

# # ============================================================
# # Remediation OTHER responses
# # ============================================================

# REM_OTHER = {
#     "NotifyAdapterChargeWattageIsNotSufficient": [
#         "Notification sent directly to the Windows Notification Center without a pop up",
#         "Notification could not be displayed because Windows notifications are disabled in system",
#         "Notification could not be displayed because no user is logged in",
#         "Notification could not be displayed because user system is locked",
#         "Failed to display AC Adapter Charge Wattage Notification.",
#         "Adapter wattage data not found in DTP.",
#         "Some error occurred while executing NotifyAdapterChargeWattageIsNotSufficient."
#     ],

#     "SetBatteryChargeCap": [
#         "Couldn't update BIOS battery charge level settings due to a script error.",
#         "Some error occurred while executing the tool RemediateBatteryChargeCap."
#     ]
# }

# ============================================================
# Diagnostic to Remediation mapping
# ============================================================

DIAG_TO_REM = {
    "GetACAdapterChargeWattageStatus":
        "NotifyAdapterChargeWattageIsNotSufficient",

    "CheckBatteryChargeCap":
        "SetBatteryChargeCap"
}

# ============================================================
# Active registry — modified per test
# ============================================================

TOOL_RESPONSES = {}

def execute_tool(tool_name):
    if tool_name in TOOL_RESPONSES:
        return json.dumps({"result": TOOL_RESPONSES[tool_name]})

    return json.dumps({"result": f"Unknown tool: {tool_name}"})


# ============================================================
# Configure fallthrough scenario
# First diagnostic NOT CONFIRMED / ERROR
# Second diagnostic CONFIRMED
# Remediation SUCCESS
# ============================================================

def setup_fallthrough(first_diag, second_diag, remediation, use_error=False):

    global TOOL_RESPONSES
    TOOL_RESPONSES = {}

    # Validate diagnostic tools
    assert first_diag in DIAG_CONFIRMED
    assert second_diag in DIAG_CONFIRMED

    # Prevent repeating the same diagnostic
    assert first_diag != second_diag, "Diagnostic tools must be different"

    # Validate remediation mapping
    assert DIAG_TO_REM[second_diag] == remediation, (
        "Remediation does not match the confirmed diagnostic"
    )

    # First diagnostic — NOT CONFIRMED or ERROR
    if use_error:
        TOOL_RESPONSES[first_diag] = DIAG_ERROR[first_diag]
        print(
            f"  [SET] {first_diag} -> ERROR: "
            f"{DIAG_ERROR[first_diag][:60]}..."
        )
    else:
        TOOL_RESPONSES[first_diag] = DIAG_NOT_CONFIRMED[first_diag]
        print(
            f"  [SET] {first_diag} -> NOT CONFIRMED: "
            f"{DIAG_NOT_CONFIRMED[first_diag][:60]}..."
        )

    # Second diagnostic — CONFIRMED
    TOOL_RESPONSES[second_diag] = DIAG_CONFIRMED[second_diag]

    print(
        f"  [SET] {second_diag} -> CONFIRMED: "
        f"{DIAG_CONFIRMED[second_diag][:60]}..."
    )

    # Remediation — SUCCESS
    TOOL_RESPONSES[remediation] = REM_SUCCESS[remediation]

    print(
        f"  [SET] {remediation} -> SUCCESS: "
        f"{REM_SUCCESS[remediation][:60]}..."
    )


print("Cell 1: Battery Not Charging Tool Registry loaded")
print("Diagnostic tools:", len(DIAG_CONFIRMED))
print("Remediation tools:", len(REM_SUCCESS))


Cell 1: Battery Not Charging Tool Registry loaded
Diagnostic tools: 2
Remediation tools: 2


In [8]:
import time
import torch

# ============================================================
# SYSTEM PROMPT — Matches your SmolLM training format
# ============================================================
SYSTEM_PROMPT = """You are the Reasoning & Execution engine for Battery Not Charging issues. Your role is to analyze SHAP feature attributions, execute diagnostics & remediations when issues are confirmed, and report the final outcome.

CORE RULES
- REACT SEQUENCE: Number each step sequentially (THOUGHT N:, ACTION N:). Do not emit a subsequent THOUGHT until you receive the output of your ACTION.
- TOOL CADENCE: Call only one tool per turn. Never repeat a diagnostic tool that has already run.
- AUTO-REMEDIATION: Once a diagnostic tool confirms a symptom, proceed immediately to invoke the corresponding remediation tool.

REASONING PROCESS
1. Hypothesis & Diagnostics:
   Select the appropriate read-only diagnostic tool to verify the root cause, and execute it.
2. Evidence Evaluation & Action:
   Read the diagnostic observation. If an issue is verified, explain the underlying root cause and execute the corresponding corrective remediation tool.
3. Resolution:
   Confirm the remediation outcome and output your final assessment.

TOOL FORMAT: {"tool": "<tool_name>"}

FINAL ANSWER FORMAT:
Root Cause: <Explanation of the verified issue>
Evidence:
- <DiagnosticToolName> returned: "<Diagnostic output>"
Remediation:
- Function: <RemediationToolName>
- Execution Result: <Remediation output>
- Status: EXECUTED"""
# ============================================================
# TOOL CALL DETECTOR — Matches {"tool": "<name>"} format
# ============================================================
TOOL_PATTERN = r'\{"tool":\s*"([^"]+)"\}'

def detect_tool_call(text):
    """Detect tool call in model output"""
    matches = list(re.finditer(TOOL_PATTERN, text))
    if matches:
        match = matches[-1]
        return {"tool_name": match.group(1), "end_pos": match.end()}
    return None

def trim_at_tool_call(text):
    """Trim text at tool call boundary"""
    match = re.search(TOOL_PATTERN, text)
    if match:
        return text[:match.end()].strip(), True
    return text.strip(), False

# ============================================================
# USER PROMPT BUILDER
# ============================================================
def to_user_prompt(data):
    """Build user prompt matching training format"""
    parts = []
    if "calibrated_confidence" in data:
        parts.append(f"calibrated_confidence: {data['calibrated_confidence']}")
    if "deviation_score" in data:
        parts.append(f"deviation_score: {data['deviation_score']}")
    if "evidence_tier" in data:
        parts.append(f"evidence_tier: {data['evidence_tier']}")
    parts.append("")
    parts.append("shap_features:")
    for f in data["shap_features"]:
        parts.append(f"- {f['feature']}: {f['shap_value']}")
    return "\n".join(parts)

# ============================================================
# MULTI-TURN INFERENCE LOOP
# Key: Tool responses come as role="user" with [TOOL_RESPONSE] prefix
# ============================================================
def run_test(test_input, model, tokenizer, max_tool_calls=5):
    """Run inference matching SmolLM training format"""
    user_prompt = to_user_prompt(test_input)

    print("=" * 60)
    print(f"  TEST: {test_input.get('name', 'Unnamed')}")
    print("=" * 60)
    print(f"  INPUT:\n{user_prompt}")
    print("=" * 60)

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt}
    ]

    tool_calls_made = 0
    start = time.time()

    while tool_calls_made < max_tool_calls:
        text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        inputs = tokenizer(text, return_tensors="pt").to("cuda")

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=200,
                do_sample=True,
                temperature=0.1,
                top_p=0.9,
                repetition_penalty=1.1
            )

        new_text = tokenizer.decode(
            outputs[0][inputs["input_ids"].shape[-1]:],
            skip_special_tokens=True
        ).strip()

        trimmed_text, has_tool_call = trim_at_tool_call(new_text)

        if has_tool_call:
            tool_call = detect_tool_call(trimmed_text)
            if tool_call:
                tool_calls_made += 1

                # Add assistant turn (ends at ACTION)
                messages.append({"role": "assistant", "content": trimmed_text})

                # Execute tool from registry
                result = execute_tool(tool_call["tool_name"])

                # Add tool response as USER with [TOOL_RESPONSE] prefix
                # This matches your SmolLM training format
                messages.append({
                    "role": "user",
                    "content": f"[TOOL_RESPONSE] {result}"
                })
            else:
                messages.append({"role": "assistant", "content": new_text})
                break
        else:
            messages.append({"role": "assistant", "content": new_text})
            break

    elapsed = time.time() - start

    # Print conversation in chronological order
    print(f"\n{'='*60}")
    print("  CONVERSATION")
    print("=" * 60)
    turn = 0
    for msg in messages:
        if msg["role"] == "system":
            continue
        elif msg["role"] == "user" and not msg["content"].startswith("[TOOL_RESPONSE]"):
            continue
        elif msg["role"] == "assistant":
            turn += 1
            print(f"\n[ASSISTANT TURN {turn}]")
            print(msg["content"])
            print("-" * 40)
        elif msg["role"] == "user" and msg["content"].startswith("[TOOL_RESPONSE]"):
            print(f"[TOOL RESULT] {msg['content']}")
            print("-" * 40)

    print("=" * 60)
    print(f"Time: {elapsed:.2f}s | Tool calls: {tool_calls_made}")
    print("=" * 60)

    return messages

print("Cell 2: Inference engine loaded successfully")

Cell 2: Inference engine loaded successfully


In [9]:
import logging
import warnings

# Suppress MatMul8bitLt warnings
warnings.filterwarnings(
    "ignore",
    message=r".*MatMul8bitLt.*",
    category=UserWarning
)

# Suppress bitsandbytes warning logs
logging.getLogger("bitsandbytes").setLevel(logging.ERROR)

In [11]:
setup_fallthrough(
    first_diag="GetACAdapterChargeWattageStatus",
    second_diag="CheckBatteryChargeCap",
    remediation="SetBatteryChargeCap"
)

messages = run_test({
    "name": "FT-A: AC Adapter NOT confirmed -> Battery Charge Cap CONFIRMED",
    # "calibrated_confidence": 0.83,
    # "deviation_score": 0.81,
    # "evidence_tier": "STRONG",
    "shap_features": [
        {"feature": "AttachedAdapterWattage", "shap_value": 0.35},
        {"feature": "BattChgMode", "shap_value": 0.28},
        {"feature": "ac_connect_disconnect_events_24h", "shap_value": 0.05}
    ]
}, model, tokenizer)


  [SET] GetACAdapterChargeWattageStatus -> NOT CONFIRMED: Sufficient AC Adapter charge wattage detected.No need to dis...
  [SET] CheckBatteryChargeCap -> CONFIRMED: Battery charge cap is enabled....
  [SET] SetBatteryChargeCap -> SUCCESS: BIOS battery charge level settings were updated. Remediation...
  TEST: FT-A: AC Adapter NOT confirmed -> Battery Charge Cap CONFIRMED
  INPUT:

shap_features:
- AttachedAdapterWattage: 0.35
- BattChgMode: 0.28
- ac_connect_disconnect_events_24h: 0.05

  CONVERSATION

[ASSISTANT TURN 1]
THOUGHT 1:
The highest SHAP feature is AttachedAdapterWattage.

Looking up AttachedAdapterWattage
→ It is EXCLUSIVE
→ It maps to the diagnostic tool: GetACAdapterChargeWattageStatus

I will call the diagnostic tool: GetACAdapterChargeWattageStatus.

ACTION 1:
{"tool":"GetACAdapterChargeWattageStatus"}
----------------------------------------
[TOOL RESULT] [TOOL_RESPONSE] {"result": "Sufficient AC Adapter charge wattage detected.No need to display AC Adapter Charge Wat

In [ ]:
# setup_fallthrough(
#     first_diag="GetACAdapterChargeWattageStatus",
#     second_diag="CheckLowEnergySettings",
#     remediation="SetLowEnergySettings"
# )

# messages = run_test({
#     "name": "FT-A: AC Adapter NOT confirmed -> Heavy Apps CONFIRMED",
#     # "calibrated_confidence": 0.83,
#     # "deviation_score": 0.81,
#     # "evidence_tier": "STRONG",
#     "shap_features": [
#         {"feature": "AttachedAdapterWattage", "shap_value": 0.35},
#         {"feature": "EppValue", "shap_value": 0.28},
#         {"feature": "ac_connect_disconnect_events_24h", "shap_value": 0.05}
#     ]
# }, model, tokenizer)

In [ ]:
# setup_fallthrough(
#     first_diag="GetACAdapterChargeWattageStatus",
#     second_diag="GetBatteryChargeStateChangesCount",
#     remediation="NotifyBatteryChargeStateChanges"
# )

# messages = run_test({
#     "name": "FT-B: AC Adapter NOT confirmed -> Frequent AC CONFIRMED",
#     # "calibrated_confidence": 0.84,
#     # "deviation_score": 0.82,
#     # "evidence_tier": "STRONG",
#     "shap_features": [
#         {"feature": "MinimumRequiredAdapterWattage", "shap_value": 0.36},
#         {"feature": "ac_connect_disconnect_events_24h", "shap_value": 0.29},
#         {"feature": "EppValue", "shap_value": 0.05}
#     ]
# }, model, tokenizer)

In [ ]:
# setup_fallthrough(
#     first_diag="GetBatteryChargeStateChangesCount",
#     second_diag="GetACAdapterChargeWattageStatus",
#     remediation="NotifyAdapterChargeWattageIsNotSufficient"
# )

# messages = run_test({
#     "name": "FT-G: Frequent AC NOT confirmed -> AC Adapter CONFIRMED",
#     # "calibrated_confidence": 0.83,
#     # "deviation_score": 0.81,
#     # "evidence_tier": "STRONG",
#     "shap_features": [
#         {"feature": "ac_connect_disconnect_events_24h", "shap_value": 0.35},
#         {"feature": "AttachedAdapterWattage", "shap_value": 0.28},
#         {"feature": "EppValue", "shap_value": 0.05}
#     ]
# }, model, tokenizer)

In [ ]:
# setup_fallthrough(
#     first_diag="UpdateDrivers",
#     second_diag="CheckLowEnergySettings",
#     remediation="SetLowEnergySettings"
# )

# messages = run_test({
#     "name": "FT-A: AC Adapter NOT confirmed -> Heavy Apps CONFIRMED",
#     # "calibrated_confidence": 0.83,
#     # "deviation_score": 0.81,
#     # "evidence_tier": "STRONG",
#     "shap_features": [
#         {"feature": "bios_microcode_revision", "shap_value": 0.35},
#         {"feature": "EppValue", "shap_value": 0.28},
#         {"feature": "ac_connect_disconnect_events_24h", "shap_value": 0.05}
#     ]
# }, model, tokenizer)

In [ ]:
# import time
# import torch
# import json
# import re

# #SYSTEM_PROMPT = """You are the Reasoning & Execution engine for Battery Quick Discharging issues. Your role is to analyze SHAP feature attributions, execute diagnostics & remediations when issues are confirmed, and report the final outcome. CORE RULES - REACT SEQUENCE: Number each step sequentially (THOUGHT N:, ACTION N:). Do not emit a subsequent THOUGHT until you receive the output of your ACTION. - TOOL CADENCE: Call only one tool per turn. Never repeat a diagnostic tool that has already run. - AUTO-REMEDIATION: Once a diagnostic tool confirms a symptom, proceed immediately to invoke the corresponding remediation tool. REASONING PROCESS 1. Hypothesis & Diagnostics: Select the appropriate read-only diagnostic tool to verify the root cause, and execute it. 2. Evidence Evaluation & Action: Read the diagnostic observation. If an issue is verified, explain the underlying root cause and execute the corresponding corrective remediation tool. 3. Resolution: Confirm the remediation outcome and output your final assessment. TOOL FORMAT: {"tool": "<tool_name>"} TOOL RESPONSE FORMAT: - After you emit an ACTION, the tool result will be provided as a [TOOL_RESPONSE] message. Parse the JSON result and continue your THOUGHT/ACTION sequence accordingly. FINAL ANSWER FORMAT: Root Cause: <Explanation of the verified issue> Evidence: - <DiagnosticToolName> returned: "<Diagnostic output>" Remediation: - Function: <RemediationToolName> - Execution Result: <Remediation output> - Status: EXECUTED"""

# SYSTEM_PROMPT = """You are the Reasoning & Execution engine for Battery Quick Discharging issues. Your role is to analyze SHAP feature attributions, execute diagnostics & remediations when issues are confirmed, and report the final outcome.

# CORE RULES
# - REACT SEQUENCE: Number each step sequentially (THOUGHT N:, ACTION N:). Do not emit a subsequent THOUGHT until you receive the output of your ACTION.
# - TOOL CADENCE: Call only one tool per turn. Never repeat a diagnostic tool that has already run.
# - AUTO-REMEDIATION: Once a diagnostic tool confirms a symptom, proceed immediately to invoke the corresponding remediation tool.

# REASONING PROCESS
# 1. Hypothesis & Diagnostics:
#    Select the appropriate read-only diagnostic tool to verify the root cause, and execute it.
# 2. Evidence Evaluation & Action:
#    Read the diagnostic observation. If an issue is verified, explain the underlying root cause and execute the corresponding corrective remediation tool.
# 3. Resolution:
#    Confirm the remediation outcome and output your final assessment.

# TOOL FORMAT: {"tool": "<tool_name>"}

# FINAL ANSWER FORMAT:
# Root Cause: <Explanation of the verified issue>
# Evidence:
# - <DiagnosticToolName> returned: "<Diagnostic output>"
# Remediation:
# - Function: <RemediationToolName>
# - Execution Result: <Remediation output>
# - Status: EXECUTED"""

# TOOL_PATTERN = r'\{"tool":\s*"([^"]+)"\}'

# def detect_tool_call(text):
#     matches = list(re.finditer(TOOL_PATTERN, text))
#     if matches:
#         match = matches[-1]
#         return {"tool_name": match.group(1), "end_pos": match.end()}
#     return None

# def trim_at_tool_call(text):
#     match = re.search(TOOL_PATTERN, text)
#     if match:
#         return text[:match.end()].strip(), True
#     return text.strip(), False

# def trim_at_final_answer(text):
#     """If FINAL ANSWER is present, keep only up to the end of the FINAL ANSWER block"""
#     if "FINAL ANSWER:" in text:
#         # Find FINAL ANSWER and keep everything up to Status: EXECUTED
#         idx = text.find("FINAL ANSWER:")
#         # Find "Status: EXECUTED" after FINAL ANSWER
#         status_idx = text.find("Status: EXECUTED", idx)
#         if status_idx != -1:
#             return text[:status_idx + len("Status: EXECUTED")].strip(), True
#         # If no Status: EXECUTED found, just keep up to end of text
#         return text.strip(), True
#     return text, False

# def run_test(test_input, model, tokenizer, max_tool_calls=6):
#     parts = []
#     if "calibrated_confidence" in test_input:
#         parts.append(f"calibrated_confidence: {test_input['calibrated_confidence']}")
#     if "deviation_score" in test_input:
#         parts.append(f"deviation_score: {test_input['deviation_score']}")
#     if "evidence_tier" in test_input:
#         parts.append(f"evidence_tier: {test_input['evidence_tier']}")
#     parts.append("")
#     parts.append("shap_features:")
#     for f in test_input["shap_features"]:
#         parts.append(f"- {f['feature']}: {f['shap_value']}")
#     user_prompt = "\n".join(parts)

#     print("=" * 60)
#     print(f"  TEST: {test_input.get('name', 'Unnamed')}")
#     print("=" * 60)
#     print(f"  INPUT:\n{user_prompt}")
#     print("=" * 60)

#     messages = [
#         {"role": "system", "content": SYSTEM_PROMPT},
#         {"role": "user", "content": user_prompt}
#     ]

#     tool_calls_made = 0
#     start = time.time()
#     reached_final = False

#     while tool_calls_made < max_tool_calls and not reached_final:
#         text = tokenizer.apply_chat_template(
#             messages, tokenize=False, add_generation_prompt=True
#         )
#         inputs = tokenizer(text, return_tensors="pt").to("cuda")

#         with torch.no_grad():
#             outputs = model.generate(
#                 **inputs,
#                 max_new_tokens=300,
#                 do_sample=True,
#                 temperature=0.2,
#                 top_p=0.9,
#                 repetition_penalty=1.2
#             )

#         new_text = tokenizer.decode(
#             outputs[0][inputs["input_ids"].shape[-1]:],
#             skip_special_tokens=True
#         ).strip()

#         # Check if FINAL ANSWER is in the output
#         final_text, has_final = trim_at_final_answer(new_text)
#         if has_final:
#             messages.append({"role": "assistant", "content": final_text})
#             reached_final = True
#             continue

#         # Check for tool call
#         trimmed_text, has_tool_call = trim_at_tool_call(new_text)

#         if has_tool_call:
#             tool_call = detect_tool_call(trimmed_text)
#             if tool_call:
#                 tool_calls_made += 1
#                 messages.append({"role": "assistant", "content": trimmed_text})
#                 result = execute_tool(tool_call["tool_name"])
#                 messages.append({
#                     "role": "user",
#                     "content": f"[TOOL_RESPONSE] {result}"
#                 })
#             else:
#                 messages.append({"role": "assistant", "content": new_text})
#                 break
#         else:
#             messages.append({"role": "assistant", "content": new_text})
#             break

#     elapsed = time.time() - start

#     print(f"\n{'='*60}")
#     print("  CONVERSATION")
#     print("=" * 60)
#     turn = 0
#     for msg in messages:
#         if msg["role"] == "system":
#             continue
#         elif msg["role"] == "user" and not msg["content"].startswith("[TOOL_RESPONSE]"):
#             continue
#         elif msg["role"] == "assistant":
#             turn += 1
#             print(f"\n[ASSISTANT TURN {turn}]")
#             print(msg["content"])
#             print("-" * 40)
#         elif msg["role"] == "user" and msg["content"].startswith("[TOOL_RESPONSE]"):
#             print(f"[TOOL RESULT] {msg['content']}")
#             print("-" * 40)

#     print("=" * 60)
#     print(f"Time: {elapsed:.2f}s | Tool calls: {tool_calls_made}")
#     if reached_final:
#         print("FINAL ANSWER detected — generation stopped correctly")
#     print("=" * 60)

#     return messages

# print("Inference engine loaded with FINAL ANSWER detection")

In [ ]:
# import time
# import torch

# SYSTEM_PROMPT = """You are the Reasoning & Execution engine for Battery Quick Discharging issues. Your role is to analyze SHAP feature attributions, execute diagnostics & remediations when issues are confirmed, and report the final outcome.

# CORE RULES
# - REACT SEQUENCE: Number each step sequentially (THOUGHT N:, ACTION N:). Do not emit a subsequent THOUGHT until you receive the output of your ACTION.
# - TOOL CADENCE: Call only one tool per turn. Never repeat a diagnostic tool that has already run.
# - AUTO-REMEDIATION: Once a diagnostic tool confirms a symptom, proceed immediately to invoke the corresponding remediation tool.

# REASONING PROCESS
# 1. Hypothesis & Diagnostics:
#    Select the appropriate read-only diagnostic tool to verify the root cause, and execute it.
# 2. Evidence Evaluation & Action:
#    Read the diagnostic observation. If an issue is verified, explain the underlying root cause and execute the corresponding corrective remediation tool.
# 3. Resolution:
#    Confirm the remediation outcome and output your final assessment.

# TOOL FORMAT: {"tool": "<tool_name>"}

# FINAL ANSWER FORMAT:
# Root Cause: <Explanation of the verified issue>
# Evidence:
# - <DiagnosticToolName> returned: "<Diagnostic output>"
# Remediation:
# - Function: <RemediationToolName>
# - Execution Result: <Remediation output>
# - Status: EXECUTED"""
# TOOL_PATTERN = r'\{"tool":\s*"([^"]+)"\}'

# def detect_tool_call(text):
#     matches = list(re.finditer(TOOL_PATTERN, text))
#     if matches:
#         match = matches[-1]
#         return {"tool_name": match.group(1), "end_pos": match.end()}
#     return None

# def trim_at_tool_call(text):
#     match = re.search(TOOL_PATTERN, text)
#     if match:
#         return text[:match.end()].strip(), True
#     return text.strip(), False

# def to_user_prompt(data):
#     parts = []
#     if "calibrated_confidence" in data:
#         parts.append(f"calibrated_confidence: {data['calibrated_confidence']}")
#     if "deviation_score" in data:
#         parts.append(f"deviation_score: {data['deviation_score']}")
#     if "evidence_tier" in data:
#         parts.append(f"evidence_tier: {data['evidence_tier']}")
#     parts.append("")
#     parts.append("shap_features:")
#     for f in data["shap_features"]:
#         parts.append(f"- {f['feature']}: {f['shap_value']}")
#     return "\n".join(parts)

# def run_test(test_input, model, tokenizer, max_tool_calls=10):
#     user_prompt = to_user_prompt(test_input)

#     print("=" * 60)
#     print(f"  TEST: {test_input.get('name', 'Unnamed')}")
#     print("=" * 60)
#     print(f"  INPUT:\n{user_prompt}")
#     print("=" * 60)

#     messages = [
#         {"role": "system", "content": SYSTEM_PROMPT},
#         {"role": "user", "content": user_prompt}
#     ]

#     tool_calls_made = 0
#     start = time.time()

#     while tool_calls_made < max_tool_calls:
#         text = tokenizer.apply_chat_template(
#             messages, tokenize=False, add_generation_prompt=True
#         )
#         inputs = tokenizer(text, return_tensors="pt").to("cuda")

#         with torch.no_grad():
#             outputs = model.generate(
#                 **inputs,
#                 max_new_tokens=200,
#                 do_sample=True,
#                 temperature=0.3,
#                 top_p=0.9,
#                 repetition_penalty=1.1
#             )

#         new_text = tokenizer.decode(
#             outputs[0][inputs["input_ids"].shape[-1]:],
#             skip_special_tokens=True
#         ).strip()

#         trimmed_text, has_tool_call = trim_at_tool_call(new_text)

#         if has_tool_call:
#             tool_call = detect_tool_call(trimmed_text)
#             if tool_call:
#                 tool_calls_made += 1
#                 messages.append({"role": "assistant", "content": trimmed_text})
#                 result = execute_tool(tool_call["tool_name"])
#                 messages.append({
#                     "role": "user",
#                     "content": f"[TOOL_RESPONSE] {result}"
#                 })
#             else:
#                 messages.append({"role": "assistant", "content": new_text})
#                 break
#         else:
#             messages.append({"role": "assistant", "content": new_text})
#             break

#     elapsed = time.time() - start

#     print(f"\n{'='*60}")
#     print("  CONVERSATION")
#     print("=" * 60)
#     turn = 0
#     for msg in messages:
#         if msg["role"] == "system":
#             continue
#         elif msg["role"] == "user" and not msg["content"].startswith("[TOOL_RESPONSE]"):
#             continue
#         elif msg["role"] == "assistant":
#             turn += 1
#             print(f"\n[ASSISTANT TURN {turn}]")
#             print(msg["content"])
#             print("-" * 40)
#         elif msg["role"] == "user" and msg["content"].startswith("[TOOL_RESPONSE]"):
#             print(f"[TOOL RESULT] {msg['content']}")
#             print("-" * 40)

#     print("=" * 60)
#     print(f"Time: {elapsed:.2f}s | Tool calls: {tool_calls_made}")
#     print("=" * 60)

#     return messages

# print("Cell 2: Inference engine loaded")

In [ ]:
# ============================================================
# FALLTHROUGH TEST CASES
# First diagnostic NOT confirmed → Second diagnostic CONFIRMED
# ============================================================

# --- Test 1: AC Adapter NOT confirmed → Heavy Apps CONFIRMED ---
test_ft_adapter_to_apps = {
    "name": "Fallthrough: AC Adapter fails -> Heavy Apps confirmed",
    # "calibrated_confidence": 0.83,
    # "deviation_score": 0.81,
    # "evidence_tier": "STRONG",
    "shap_features": [
        {"feature": "AttachedAdapterWattage", "shap_value": 0.35},
        {"feature": "AppName", "shap_value": 0.28},
        {"feature": "ac_connect_disconnect_events_24h", "shap_value": 0.05}
    ]
}

# --- Test 2: AC Adapter NOT confirmed → Frequent AC CONFIRMED ---
test_ft_adapter_to_freq = {
    "name": "Fallthrough: AC Adapter fails -> Frequent AC confirmed",
    # "calibrated_confidence": 0.84,
    # "deviation_score": 0.82,
    # "evidence_tier": "STRONG",
    "shap_features": [
        {"feature": "MinimumRequiredAdapterWattage", "shap_value": 0.36},
        {"feature": "ac_connect_disconnect_events_24h", "shap_value": 0.29},
        {"feature": "EppValue", "shap_value": 0.05}
    ]
}

# --- Test 3: AC Adapter NOT confirmed → Energy Settings CONFIRMED ---
test_ft_adapter_to_energy = {
    "name": "Fallthrough: AC Adapter fails -> Energy Settings confirmed",
    # "calibrated_confidence": 0.82,
    # "deviation_score": 0.80,
    # "evidence_tier": "STRONG",
    "shap_features": [
        {"feature": "AttachedAdapterWattage", "shap_value": 0.37},
        {"feature": "EppValue", "shap_value": 0.27},
        {"feature": "bios_full_version", "shap_value": 0.05}
    ]
}

# --- Test 4: AC Adapter NOT confirmed → Outdated Driver CONFIRMED ---
test_ft_adapter_to_driver = {
    "name": "Fallthrough: AC Adapter fails -> Outdated Driver confirmed",
    # "calibrated_confidence": 0.83,
    # "deviation_score": 0.81,
    # "evidence_tier": "STRONG",
    "shap_features": [
        {"feature": "AttachedAdapterWattage", "shap_value": 0.34},
        {"feature": "bios_full_version", "shap_value": 0.28},
        {"feature": "ac_connect_disconnect_events_24h", "shap_value": 0.05}
    ]
}

# --- Test 5: Heavy Apps NOT confirmed → AC Adapter CONFIRMED ---
test_ft_apps_to_adapter = {
    "name": "Fallthrough: Heavy Apps fails -> AC Adapter confirmed",
    # "calibrated_confidence": 0.82,
    # "deviation_score": 0.80,
    # "evidence_tier": "STRONG",
    "shap_features": [
        {"feature": "AppName", "shap_value": 0.35},
        {"feature": "AttachedAdapterWattage", "shap_value": 0.28},
        {"feature": "EppValue", "shap_value": 0.05}
    ]
}

# --- Test 6: Heavy Apps NOT confirmed → Frequent AC CONFIRMED ---
test_ft_apps_to_freq = {
    "name": "Fallthrough: Heavy Apps fails -> Frequent AC confirmed",
    # "calibrated_confidence": 0.81,
    # "deviation_score": 0.79,
    # "evidence_tier": "STRONG",
    "shap_features": [
        {"feature": "CpuEnergy", "shap_value": 0.34},
        {"feature": "ac_connect_disconnect_events_24h", "shap_value": 0.27},
        {"feature": "bios_full_version", "shap_value": 0.05}
    ]
}

# --- Test 7: Heavy Apps NOT confirmed (error) → Energy Settings CONFIRMED ---
test_ft_apps_error_to_energy = {
    "name": "Fallthrough: Heavy Apps error -> Energy Settings confirmed",
    # "calibrated_confidence": 0.83,
    # "deviation_score": 0.81,
    # "evidence_tier": "STRONG",
    "shap_features": [
        {"feature": "AppName", "shap_value": 0.36},
        {"feature": "EppValue", "shap_value": 0.28},
        {"feature": "ac_connect_disconnect_events_24h", "shap_value": 0.04}
    ]
}

# --- Test 8: Frequent AC NOT confirmed → AC Adapter CONFIRMED ---
test_ft_freq_to_adapter = {
    "name": "Fallthrough: Frequent AC fails -> AC Adapter confirmed",
    # "calibrated_confidence": 0.83,
    # "deviation_score": 0.81,
    # "evidence_tier": "STRONG",
    "shap_features": [
        {"feature": "ac_connect_disconnect_events_24h", "shap_value": 0.35},
        {"feature": "AttachedAdapterWattage", "shap_value": 0.28},
        {"feature": "EppValue", "shap_value": 0.05}
    ]
}

# --- Test 9: Frequent AC NOT confirmed → Heavy Apps CONFIRMED ---
test_ft_freq_to_apps = {
    "name": "Fallthrough: Frequent AC fails -> Heavy Apps confirmed",
    # "calibrated_confidence": 0.82,
    # "deviation_score": 0.80,
    # "evidence_tier": "STRONG",
    "shap_features": [
        {"feature": "ac_connect_disconnect_events_24h", "shap_value": 0.34},
        {"feature": "AppName", "shap_value": 0.27},
        {"feature": "bios_full_version", "shap_value": 0.04}
    ]
}

# --- Test 10: Energy Settings NOT confirmed → AC Adapter CONFIRMED ---
test_ft_energy_to_adapter = {
    "name": "Fallthrough: Energy Settings fails -> AC Adapter confirmed",
    # "calibrated_confidence": 0.82,
    # "deviation_score": 0.80,
    # "evidence_tier": "STRONG",
    "shap_features": [
        {"feature": "EppValue", "shap_value": 0.36},
        {"feature": "AttachedAdapterWattage", "shap_value": 0.27},
        {"feature": "ac_connect_disconnect_events_24h", "shap_value": 0.05}
    ]
}

# --- Test 11: Energy Settings NOT confirmed (not on battery) → Frequent AC CONFIRMED ---
test_ft_energy_notbattery_to_freq = {
    "name": "Fallthrough: Energy Settings not on battery -> Frequent AC confirmed",
    # "calibrated_confidence": 0.83,
    # "deviation_score": 0.81,
    # "evidence_tier": "STRONG",
    "shap_features": [
        {"feature": "EppValue", "shap_value": 0.35},
        {"feature": "ac_connect_disconnect_events_24h", "shap_value": 0.28},
        {"feature": "bios_full_version", "shap_value": 0.04}
    ]
}

# --- Test 12: Outdated Driver NOT confirmed → Energy Settings CONFIRMED ---
test_ft_driver_to_energy = {
    "name": "Fallthrough: Outdated Driver fails -> Energy Settings confirmed",
    # "calibrated_confidence": 0.82,
    # "deviation_score": 0.80,
    # "evidence_tier": "STRONG",
    "shap_features": [
        {"feature": "bios_full_version", "shap_value": 0.35},
        {"feature": "EppValue", "shap_value": 0.27},
        {"feature": "ac_connect_disconnect_events_24h", "shap_value": 0.05}
    ]
}

# --- Test 13: Outdated Driver NOT confirmed (scan error) → Heavy Apps CONFIRMED ---
test_ft_driver_error_to_apps = {
    "name": "Fallthrough: Outdated Driver scan error -> Heavy Apps confirmed",
    # "calibrated_confidence": 0.82,
    # "deviation_score": 0.80,
    # "evidence_tier": "STRONG",
    "shap_features": [
        {"feature": "release_date", "shap_value": 0.34},
        {"feature": "AppName", "shap_value": 0.28},
        {"feature": "EppValue", "shap_value": 0.04}
    ]
}

# --- Test 14: Frequent AC NOT confirmed → Heavy AC CONFIRMED (SHARED feature) ---
test_ft_freq_to_heavy_ac = {
    "name": "Fallthrough: Frequent AC fails -> Heavy AC confirmed (SHARED StateChange)",
    # "calibrated_confidence": 0.82,
    # "deviation_score": 0.80,
    # "evidence_tier": "STRONG",
    "shap_features": [
        {"feature": "ac_connect_disconnect_events_24h", "shap_value": 0.33},
        {"feature": "StateChange", "shap_value": 0.28},
        {"feature": "AppName", "shap_value": 0.04}
    ]
}

# --- Test 15: Energy Settings NOT confirmed → Outdated Driver CONFIRMED ---
test_ft_energy_to_driver = {
    "name": "Fallthrough: Energy Settings fails -> Outdated Driver confirmed",
    # "calibrated_confidence": 0.82,
    # "deviation_score": 0.80,
    # "evidence_tier": "STRONG",
    "shap_features": [
        {"feature": "EppValue", "shap_value": 0.34},
        {"feature": "bios_microcode_revision", "shap_value": 0.28},
        {"feature": "ac_connect_disconnect_events_24h", "shap_value": 0.05}
    ]
}

print("Cell 3: Fallthrough test cases loaded")
print(f"  Available tests: 15")

In [ ]:
# ============================================================
# FIX: CONFIGURE FALLTHROUGH TOOL RESPONSES
# ============================================================

def configure_fallthrough(
    first_diag,
    first_msg,
    second_diag,
    second_msg,
    remediation,
    remediation_msg
):
    global TOOL_RESPONSES

    TOOL_RESPONSES = {
        first_diag: first_msg,
        second_diag: second_msg,
        remediation: remediation_msg
    }

    print("Fallthrough configured successfully")
    print("First diagnostic:", first_diag)
    print("Second diagnostic:", second_diag)
    print("Remediation:", remediation)

In [ ]:
# ============================================================
# NOT CONFIRMED messages from Tool Outputs [1]
# ============================================================
NOT_CONFIRMED_MSGS = {
    "GetACAdapterChargeWattageStatus": "Insufficient AC Adapter charge wattage detected. AC Adapter Charge Wattage Notification has to be displayed",
    "CheckHeavyBatteryUsageByApps": "Battery usage is below the threshold.",
    "UpdateDrivers": "No drivers found for installation after driver scan.",
    "CheckHeavyACAdapterUsageSettings": "Device is not primarily used on AC adapter. No need to display Battery charge state notification.",
    "GetBatteryChargeStateChangesCount": "No frequent AC Power Connect/Disconnect detected.",
    "CheckLowEnergySettings": "Low energy settings are enabled."
}

# ERROR messages from Tool Outputs [1]
ERROR_MSGS = {
    "GetACAdapterChargeWattageStatus": "Adapter wattage data not found in DTP. No need to display AC Adapter Charge Wattage Notification.",
    "CheckHeavyBatteryUsageByApps": "Some error occurred while executing the script.",
    "UpdateDrivers": "Failed to scan drivers.",
    "CheckHeavyACAdapterUsageSettings": "Some error occurred while collecting AC adapter usage data. No need to display Battery charge state notification.",
    "GetBatteryChargeStateChangesCount": "Some error occurred while executing GetBatteryChargeStateChangesCount. No need to display Battery charge state notification.",
    "CheckLowEnergySettings": "Device is not on battery."
}

# CONFIRMED messages from Tool Outputs [1]
CONFIRMED_MSGS = {
    "GetACAdapterChargeWattageStatus": "Insufficient AC Adapter charge wattage detected. AC Adapter Charge Wattage Notification has to be displayed",
    "CheckHeavyBatteryUsageByApps": "High battery usage was detected.",
    "UpdateDrivers": "Driver update has completed.",
    "CheckHeavyACAdapterUsageSettings": "Device is primarily used on AC adapter. Remediation is required.",
    "GetBatteryChargeStateChangesCount": "Frequent AC Power Connect/Disconnect.",
    "CheckLowEnergySettings": "Low energy settings are disabled."
}

# REMEDIATION SUCCESS messages from Tool Outputs [1]
REMEDIATION_SUCCESS_MSGS = {
    "NotifyAdapterChargeWattageIsNotSufficient": "AC Adapter Charge Wattage Notification displayed successfully to user.",
    "NotifyHeavyBatteryUsageByApps": "User notified of top three apps consuming battery.",
    "UpdateBIOSBatteryProfile": "BIOS battery profile has been updated to primarily on AC power.",
    "NotifyBatteryChargeStateChanges": "Battery Charge State Notification displayed successfully to user.",
    "SetLowEnergySettings": "All settings for low energy are optimized."
}

# ============================================================
# Fallthrough configurations: (first_diag, second_diag, remediation)
# ============================================================
fallthrough_configs = {
    "test_ft_adapter_to_apps": {
        "test": test_ft_adapter_to_apps,
        "first_diag": "GetACAdapterChargeWattageStatus",
        "first_msg": NOT_CONFIRMED_MSGS["GetACAdapterChargeWattageStatus"],
        "second_diag": "CheckHeavyBatteryUsageByApps",
        "second_msg": CONFIRMED_MSGS["CheckHeavyBatteryUsageByApps"],
        "remediation": "NotifyHeavyBatteryUsageByApps",
        "remediation_msg": REMEDIATION_SUCCESS_MSGS["NotifyHeavyBatteryUsageByApps"]
    },
    "test_ft_adapter_to_freq": {
        "test": test_ft_adapter_to_freq,
        "first_diag": "GetACAdapterChargeWattageStatus",
        "first_msg": NOT_CONFIRMED_MSGS["GetACAdapterChargeWattageStatus"],
        "second_diag": "GetBatteryChargeStateChangesCount",
        "second_msg": CONFIRMED_MSGS["GetBatteryChargeStateChangesCount"],
        "remediation": "NotifyBatteryChargeStateChanges",
        "remediation_msg": REMEDIATION_SUCCESS_MSGS["NotifyBatteryChargeStateChanges"]
    },
    "test_ft_adapter_to_energy": {
        "test": test_ft_adapter_to_energy,
        "first_diag": "GetACAdapterChargeWattageStatus",
        "first_msg": NOT_CONFIRMED_MSGS["GetACAdapterChargeWattageStatus"],
        "second_diag": "CheckLowEnergySettings",
        "second_msg": CONFIRMED_MSGS["CheckLowEnergySettings"],
        "remediation": "SetLowEnergySettings",
        "remediation_msg": REMEDIATION_SUCCESS_MSGS["SetLowEnergySettings"]
    },
    "test_ft_adapter_to_driver": {
        "test": test_ft_adapter_to_driver,
        "first_diag": "GetACAdapterChargeWattageStatus",
        "first_msg": NOT_CONFIRMED_MSGS["GetACAdapterChargeWattageStatus"],
        "second_diag": "UpdateDrivers",
        "second_msg": CONFIRMED_MSGS["UpdateDrivers"],
        "remediation": "UpdateDrivers",
        "remediation_msg": CONFIRMED_MSGS["UpdateDrivers"]
    },
    "test_ft_apps_to_adapter": {
        "test": test_ft_apps_to_adapter,
        "first_diag": "CheckHeavyBatteryUsageByApps",
        "first_msg": NOT_CONFIRMED_MSGS["CheckHeavyBatteryUsageByApps"],
        "second_diag": "GetACAdapterChargeWattageStatus",
        "second_msg": CONFIRMED_MSGS["GetACAdapterChargeWattageStatus"],
        "remediation": "NotifyAdapterChargeWattageIsNotSufficient",
        "remediation_msg": REMEDIATION_SUCCESS_MSGS["NotifyAdapterChargeWattageIsNotSufficient"]
    },
    "test_ft_apps_to_freq": {
        "test": test_ft_apps_to_freq,
        "first_diag": "CheckHeavyBatteryUsageByApps",
        "first_msg": NOT_CONFIRMED_MSGS["CheckHeavyBatteryUsageByApps"],
        "second_diag": "GetBatteryChargeStateChangesCount",
        "second_msg": CONFIRMED_MSGS["GetBatteryChargeStateChangesCount"],
        "remediation": "NotifyBatteryChargeStateChanges",
        "remediation_msg": REMEDIATION_SUCCESS_MSGS["NotifyBatteryChargeStateChanges"]
    },
    "test_ft_apps_error_to_energy": {
        "test": test_ft_apps_error_to_energy,
        "first_diag": "CheckHeavyBatteryUsageByApps",
        "first_msg": ERROR_MSGS["CheckHeavyBatteryUsageByApps"],
        "second_diag": "CheckLowEnergySettings",
        "second_msg": CONFIRMED_MSGS["CheckLowEnergySettings"],
        "remediation": "SetLowEnergySettings",
        "remediation_msg": REMEDIATION_SUCCESS_MSGS["SetLowEnergySettings"]
    },
    "test_ft_freq_to_adapter": {
        "test": test_ft_freq_to_adapter,
        "first_diag": "GetBatteryChargeStateChangesCount",
        "first_msg": NOT_CONFIRMED_MSGS["GetBatteryChargeStateChangesCount"],
        "second_diag": "GetACAdapterChargeWattageStatus",
        "second_msg": CONFIRMED_MSGS["GetACAdapterChargeWattageStatus"],
        "remediation": "NotifyAdapterChargeWattageIsNotSufficient",
        "remediation_msg": REMEDIATION_SUCCESS_MSGS["NotifyAdapterChargeWattageIsNotSufficient"]
    },
    "test_ft_freq_to_apps": {
        "test": test_ft_freq_to_apps,
        "first_diag": "GetBatteryChargeStateChangesCount",
        "first_msg": NOT_CONFIRMED_MSGS["GetBatteryChargeStateChangesCount"],
        "second_diag": "CheckHeavyBatteryUsageByApps",
        "second_msg": CONFIRMED_MSGS["CheckHeavyBatteryUsageByApps"],
        "remediation": "NotifyHeavyBatteryUsageByApps",
        "remediation_msg": REMEDIATION_SUCCESS_MSGS["NotifyHeavyBatteryUsageByApps"]
    },
    "test_ft_energy_to_adapter": {
        "test": test_ft_energy_to_adapter,
        "first_diag": "CheckLowEnergySettings",
        "first_msg": NOT_CONFIRMED_MSGS["CheckLowEnergySettings"],
        "second_diag": "GetACAdapterChargeWattageStatus",
        "second_msg": CONFIRMED_MSGS["GetACAdapterChargeWattageStatus"],
        "remediation": "NotifyAdapterChargeWattageIsNotSufficient",
        "remediation_msg": REMEDIATION_SUCCESS_MSGS["NotifyAdapterChargeWattageIsNotSufficient"]
    },
    "test_ft_energy_notbattery_to_freq": {
        "test": test_ft_energy_notbattery_to_freq,
        "first_diag": "CheckLowEnergySettings",
        "first_msg": ERROR_MSGS["CheckLowEnergySettings"],
        "second_diag": "GetBatteryChargeStateChangesCount",
        "second_msg": CONFIRMED_MSGS["GetBatteryChargeStateChangesCount"],
        "remediation": "NotifyBatteryChargeStateChanges",
        "remediation_msg": REMEDIATION_SUCCESS_MSGS["NotifyBatteryChargeStateChanges"]
    },
    "test_ft_driver_to_energy": {
        "test": test_ft_driver_to_energy,
        "first_diag": "UpdateDrivers",
        "first_msg": NOT_CONFIRMED_MSGS["UpdateDrivers"],
        "second_diag": "CheckLowEnergySettings",
        "second_msg": CONFIRMED_MSGS["CheckLowEnergySettings"],
        "remediation": "SetLowEnergySettings",
        "remediation_msg": REMEDIATION_SUCCESS_MSGS["SetLowEnergySettings"]
    },
    "test_ft_driver_error_to_apps": {
        "test": test_ft_driver_error_to_apps,
        "first_diag": "UpdateDrivers",
        "first_msg": ERROR_MSGS["UpdateDrivers"],
        "second_diag": "CheckHeavyBatteryUsageByApps",
        "second_msg": CONFIRMED_MSGS["CheckHeavyBatteryUsageByApps"],
        "remediation": "NotifyHeavyBatteryUsageByApps",
        "remediation_msg": REMEDIATION_SUCCESS_MSGS["NotifyHeavyBatteryUsageByApps"]
    },
    "test_ft_freq_to_heavy_ac": {
        "test": test_ft_freq_to_heavy_ac,
        "first_diag": "GetBatteryChargeStateChangesCount",
        "first_msg": NOT_CONFIRMED_MSGS["GetBatteryChargeStateChangesCount"],
        "second_diag": "CheckHeavyACAdapterUsageSettings",
        "second_msg": CONFIRMED_MSGS["CheckHeavyACAdapterUsageSettings"],
        "remediation": "UpdateBIOSBatteryProfile",
        "remediation_msg": REMEDIATION_SUCCESS_MSGS["UpdateBIOSBatteryProfile"]
    },
    "test_ft_energy_to_driver": {
        "test": test_ft_energy_to_driver,
        "first_diag": "CheckLowEnergySettings",
        "first_msg": NOT_CONFIRMED_MSGS["CheckLowEnergySettings"],
        "second_diag": "UpdateDrivers",
        "second_msg": CONFIRMED_MSGS["UpdateDrivers"],
        "remediation": "UpdateDrivers",
        "remediation_msg": CONFIRMED_MSGS["UpdateDrivers"]
    }
}

# ============================================================
# RUN ALL FALLTHROUGH TESTS
# ============================================================
results = {}

for test_name, config in fallthrough_configs.items():
    print(f"\n>>> Configuring: {test_name}")
    configure_fallthrough(
        config["first_diag"], config["first_msg"],
        config["second_diag"], config["second_msg"],
        config["remediation"], config["remediation_msg"]
    )
    messages = run_test(config["test"], model, tokenizer)
    results[test_name] = messages
    print("\n")

# ============================================================
# SUMMARY
# ============================================================
print("=" * 60)
print("  FALLTHROUGH TEST SUMMARY")
print("=" * 60)
for name, msgs in results.items():
    tool_count = sum(1 for m in msgs
                     if m["role"] == "user"
                     and m["content"].startswith("[TOOL_RESPONSE]"))
    has_final = any("FINAL ANSWER" in m["content"]
                    for m in msgs if m["role"] == "assistant")
    status = "PASS" if has_final and tool_count >= 3 else "CHECK"
    print(f"  {status} | {name} | Tool calls: {tool_count}")
print("=" * 60)